# TP6 — Couche Gold : features temporelles et label ML

Ce notebook construit `gold_machine_hourly_feature` à partir de la couche Silver :
- **1 ligne par (machine × heure)**
- **Features** : agrégats roulants 24h sur température et pression, comptage d'incidents
- **Label** : `label_failure_next_24h` — incident critique dans les 24h suivantes
- **Split temporel** : `split_set` ∈ {train, validation, test} — sans data leakage


## 0. Connexion

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import pandas as pd
import numpy as np
from datetime import datetime, timezone

db_url = URL.create(
    drivername='postgresql+psycopg2',
    username='indusense_user',
    password='ThEP@ssW0rd',
    host='localhost',
    port=5432,
    database='indusense_db',
)
engine = create_engine(db_url)
with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar())


## 1. Lecture Silver

On exclut les valeurs manquantes et les doublons pour les features numériques.


In [ ]:
df_sensors = pd.read_sql(text("""
    SELECT machine_id, observed_at, sensor_type, sensor_value
    FROM silver_sensor_reading
    WHERE is_missing = false AND is_duplicate = false
"""), engine, parse_dates=['observed_at'])

df_incidents = pd.read_sql(text("""
    SELECT machine_id, occurred_at, severity, is_label_event
    FROM silver_incident
"""), engine, parse_dates=['occurred_at'])

print(f'silver_sensor_reading (valide) : {len(df_sensors):,} lignes')
print(f'silver_incident                : {len(df_incidents):,} lignes')
print(f'Période capteurs : {df_sensors["observed_at"].min()} → {df_sensors["observed_at"].max()}')


## 2. Pivot capteurs → format large

On repasse en format large (1 ligne par machine × heure) pour calculer les features.


In [ ]:
df_wide = df_sensors.pivot_table(
    index=['machine_id', 'observed_at'],
    columns='sensor_type',
    values='sensor_value',
    aggfunc='mean'
).reset_index()
df_wide.columns.name = None

# S'assurer que les colonnes capteurs existent
for col in ['temperature_c', 'pressure_bar']:
    if col not in df_wide.columns:
        df_wide[col] = float('nan')

df_wide = df_wide.sort_values(['machine_id', 'observed_at']).reset_index(drop=True)
print(f'Format large : {len(df_wide):,} lignes')
display(df_wide.head(3))


## 3. Features roulantes 24h

Pour chaque machine, on calcule des statistiques sur une fenêtre glissante de **24 heures** (24 relevés).
La fenêtre est passée (`closed='left'`) : elle n'inclut pas l'heure courante.


In [ ]:
chunks = []
for machine, group in df_wide.groupby('machine_id'):
    group = group.sort_values('observed_at').copy()
    roll = group.set_index('observed_at')[['temperature_c', 'pressure_bar']].rolling(
        window='24h', closed='left', min_periods=1
    )
    group['temp_mean_24h']     = roll['temperature_c'].mean().values
    group['temp_max_24h']      = roll['temperature_c'].max().values
    group['temp_std_24h']      = roll['temperature_c'].std().values
    group['pressure_mean_24h'] = roll['pressure_bar'].mean().values
    group['pressure_std_24h']  = roll['pressure_bar'].std().values
    chunks.append(group)

df_features = pd.concat(chunks, ignore_index=True)
print(f'Features roulantes calculées : {len(df_features):,} lignes')
display(df_features[['machine_id','observed_at','temp_mean_24h','temp_max_24h','pressure_mean_24h']].head(3))

## 4. Features incidents (24h précédentes)

Pour chaque (machine, heure), on compte les incidents survenus dans les 24h précédentes.


In [ ]:
df_features['incident_count_prev_24h']        = 0
df_features['incident_max_severity_prev_24h'] = 0

for machine in df_features['machine_id'].unique():
    mask_f = df_features['machine_id'] == machine
    hours  = df_features.loc[mask_f, 'observed_at']
    evts   = df_incidents[df_incidents['machine_id'] == machine]
    if evts.empty:
        continue
    for idx, h in hours.items():
        window = evts[
            (evts['occurred_at'] >= h - pd.Timedelta(hours=24)) &
            (evts['occurred_at'] < h)
        ]
        df_features.at[idx, 'incident_count_prev_24h']        = len(window)
        df_features.at[idx, 'incident_max_severity_prev_24h'] = int(window['severity'].max()) if len(window) else 0

print('incident_count_prev_24h > 0 :', (df_features['incident_count_prev_24h'] > 0).sum())


## 5. Label — failure dans les 24h suivantes

`label_failure_next_24h = True` si au moins un `is_label_event=True` survient
dans les 24h qui suivent l'heure courante pour cette machine.


In [ ]:
df_features['label_failure_next_24h'] = False
label_events = df_incidents[df_incidents['is_label_event']]

for _, evt in label_events.iterrows():
    mask = (
        (df_features['machine_id'] == evt['machine_id']) &
        (df_features['observed_at'] >= evt['occurred_at'] - pd.Timedelta(hours=24)) &
        (df_features['observed_at'] <  evt['occurred_at'])
    )
    df_features.loc[mask, 'label_failure_next_24h'] = True

n_pos = df_features['label_failure_next_24h'].sum()
n_tot = len(df_features)
print(f'label=True  : {n_pos:,} ({n_pos/n_tot*100:.1f}%)')
print(f'label=False : {n_tot - n_pos:,} ({(n_tot-n_pos)/n_tot*100:.1f}%)')


## 6. Split temporel (sans data leakage)

Le split est **global sur le temps**, pas aléatoire — pour éviter le data leakage.
- `train`      : 70 % des heures les plus anciennes
- `validation` : 15 % suivantes
- `test`       : 15 % les plus récentes


In [ ]:
dates = df_features['observed_at'].sort_values().unique()
n = len(dates)
cut_train = dates[int(n * 0.70)]
cut_val   = dates[int(n * 0.85)]

def assign_split(ts):
    if ts < cut_train:
        return 'train'
    elif ts < cut_val:
        return 'validation'
    return 'test'

df_features['split_set'] = df_features['observed_at'].map(assign_split)

print(df_features.groupby('split_set')['observed_at'].agg(['min','max','count']))
print()
print('Label par split :')
print(df_features.groupby('split_set')['label_failure_next_24h'].mean().round(3))


## 7. Fenêtre temporelle et batch Gold

In [ ]:
df_features['window_start'] = df_features['observed_at']
df_features['window_end']   = df_features['observed_at'] + pd.Timedelta(hours=1)

started_at = datetime.now(timezone.utc)
with engine.begin() as conn:
    batch_gold = conn.execute(text("""
        INSERT INTO ingestion_batch (source_name, source_file, started_at, status)
        VALUES ('gold', 'silver_sensor_reading + silver_incident', :ts, 'running')
        RETURNING ingestion_batch_id
    """), {'ts': started_at}).scalar()

df_features['ingestion_batch_id'] = str(batch_gold)
print(f'Batch Gold : {batch_gold}')


## 8. Ingestion gold_machine_hourly_feature

In [ ]:
GOLD_COLS = [
    'ingestion_batch_id', 'machine_id', 'window_start', 'window_end',
    'temp_mean_24h', 'temp_max_24h', 'temp_std_24h',
    'pressure_mean_24h', 'pressure_std_24h',
    'incident_count_prev_24h', 'incident_max_severity_prev_24h',
    'label_failure_next_24h', 'split_set',
]

with engine.begin() as conn:
    conn.execute(text('TRUNCATE TABLE gold_machine_hourly_feature RESTART IDENTITY'))

df_features[GOLD_COLS].to_sql(
    'gold_machine_hourly_feature', engine,
    if_exists='append', index=False, method='multi', chunksize=2000
)
print(f'gold_machine_hourly_feature : {len(df_features):,} lignes insérées')


## 9. Clôture du batch Gold

In [ ]:
finished_at = datetime.now(timezone.utc)
n_pos = int(df_features['label_failure_next_24h'].sum())

with engine.begin() as conn:
    conn.execute(text("""
        UPDATE ingestion_batch SET
            finished_at   = :fin,
            rows_read     = :read,
            rows_loaded   = :loaded,
            rows_rejected = 0,
            status        = 'done'
        WHERE ingestion_batch_id = :bid
    """), {'fin': finished_at, 'read': len(df_features),
           'loaded': len(df_features), 'bid': str(batch_gold)})

print('Batch Gold clôturé')


## 10. Vérification finale

In [ ]:
display(pd.read_sql(text("""
    SELECT source_name, rows_read, rows_loaded, status
    FROM ingestion_batch
    WHERE source_name = 'gold'
    ORDER BY started_at DESC LIMIT 1
"""), engine))

print('--- Distribution par split ---')
display(pd.read_sql(text("""
    SELECT split_set,
           COUNT(*)                                    AS nb_lignes,
           COUNT(DISTINCT machine_id)                  AS nb_machines,
           MIN(window_start)                           AS debut,
           MAX(window_start)                           AS fin,
           ROUND(AVG(label_failure_next_24h::int)::numeric * 100, 1) AS pct_label_true
    FROM gold_machine_hourly_feature
    GROUP BY split_set
    ORDER BY debut
"""), engine))

print('--- Apercu Gold ---')
display(pd.read_sql(text("""
    SELECT machine_id, window_start, temp_mean_24h, pressure_mean_24h,
           incident_count_prev_24h, label_failure_next_24h, split_set
    FROM gold_machine_hourly_feature
    ORDER BY window_start
    LIMIT 5
"""), engine))
